# Notebook 03: GraphRAG Query
## Map-Reduce over Community Summaries

This is where GraphRAG earns its name. We pose a **global sensemaking question**:

> *"What are the main storylines and themes of the 2022 FIFA World Cup?"*

This question has no single correct chunk to retrieve — it requires synthesising information
spread across the entire document. Standard vector RAG will return the three most similar
paragraphs and miss the big picture. GraphRAG answers it by asking each community to
contribute a partial answer, then merging them.

**Paper reference:** Edge et al. (2025) — §3.1.6, Appendix E.3–E.4.

### What this notebook does

```
wc2022_trimmed_summaries.json   (15 community reports)
           │
           ▼  MAP  (App. E.4 — Community Answer Generation)
           │    For each community: LLM scores helpfulness 0–100
           │    and generates a partial answer.
           │    Filter out score = 0.
           │
           ▼  REDUCE  (App. E.4 — Global Answer Generation)
           │    Sort partial answers by score (desc).
           │    LLM synthesises a final markdown answer.
           ▼
       GraphRAG Answer
           │
           ▼  COMPARISON
       Naive RAG Answer  (TF-IDF top-3 chunks → LLM)
```

## Imports and Setup

In [ ]:
import json
import re
import subprocess
from pathlib import Path

import fitz
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import Markdown, display

print("Imports OK")

In [ ]:
import os
import platform

PDF_STEM      = "wc2022_trimmed"
PDF_PATH      = Path("../data/input")  / f"{PDF_STEM}.pdf"
SUMMARIES_PATH = Path("../data/output") / f"{PDF_STEM}_summaries.json"

QUESTION = "What are the main storylines and themes of the 2022 FIFA World Cup?"


def ollama_base_url() -> str:
    """Locate the Ollama server so this notebook runs on any machine.

    Priority: OLLAMA_BASE_URL env var  →  WSL2 Windows-host IP  →  localhost.
    Set OLLAMA_BASE_URL yourself if Ollama runs somewhere else (e.g. a remote box).
    """
    if os.environ.get("OLLAMA_BASE_URL"):
        return os.environ["OLLAMA_BASE_URL"]
    if "microsoft" in platform.uname().release.lower():   # WSL2 → Ollama on the Windows host
        host = subprocess.check_output(
            "ip route list default | awk '{print $3}'", shell=True
        ).decode().strip()
        if host:
            return f"http://{host}:11434"
    return "http://localhost:11434"                        # native Linux / macOS / Windows

BASE_URL = ollama_base_url()

llm = ChatOllama(
    model="qwen2.5:7b",
    base_url=BASE_URL,
    temperature=0.0,
    num_predict=1500,
)

print(f"Question  : {QUESTION}")
print(f"LLM       : qwen2.5:7b  @  {BASE_URL}")

---
## Step 1 — Load Community Summaries

In [ ]:
with open(SUMMARIES_PATH, encoding="utf-8") as f:
    summaries = json.load(f)

valid = [s for s in summaries if "_error" not in s and s.get("summary")]
print(f"Loaded {len(summaries)} summaries, {len(valid)} valid.")
print()
for s in valid:
    print(f"  [{s['community_id']}] {s.get('rating', 0):>4.1f} — {s.get('title', '?')[:60]}")

---
## Step 2 — Map: Score Each Community  *(Appendix E.4)*

For every community summary we ask the LLM two things at once:

1. **How helpful is this community for answering the question?** (score 0–100)  
2. **Write a partial answer** based only on this community's content.

The helpfulness score is the key innovation — it lets us automatically rank which
communities are relevant before the Reduce step, without manual filtering.
Communities that score 0 are discarded entirely.

The prompt template (Appendix E.4):

In [ ]:
MAP_PROMPT = """\
---Goal---
Generate a response to the user's question based on the community report below.
At the very start of your response, output an integer score 0-100 indicating how
helpful this community report is for answering the question.
Use the format: <ANSWER_HELPFULNESS>score</ANSWER_HELPFULNESS>

Score 0 means the report is not relevant at all.
Score 100 means the report directly and comprehensively answers the question.

---Question---
{question}

---Community Report---
{community_report}

Output:"""


def format_summary(s: dict) -> str:
    """Convert a JSON summary into readable text for the LLM."""
    lines = [
        f"Title: {s.get('title', '')}",
        f"Summary: {s.get('summary', '')}",
        "Key findings:",
    ]
    for f in s.get("findings", []):
        lines.append(f"  - {f.get('summary', '')}: {f.get('explanation', '')}")
    return "\n".join(lines)


def parse_map_response(text: str) -> tuple[int, str]:
    m = re.search(r'<ANSWER_HELPFULNESS>(\d+)</ANSWER_HELPFULNESS>', text)
    score = int(m.group(1)) if m else 0
    answer = re.sub(r'<ANSWER_HELPFULNESS>\d+</ANSWER_HELPFULNESS>\s*', '', text).strip()
    return score, answer


print("Map helpers defined.")

In [ ]:
print(f"Running Map step over {len(valid)} communities ...\n")

map_results = []

for s in valid:
    prompt = MAP_PROMPT.format(
        question=QUESTION,
        community_report=format_summary(s),
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    score, answer = parse_map_response(response.content)
    map_results.append({"community_id": s["community_id"], "score": score, "answer": answer})
    print(f"  [{s['community_id']}] score={score:>3}  {s.get('title','')[:50]}")

print(f"\nMap complete.")

In [ ]:
# Filter score=0, sort descending
useful = [r for r in map_results if r["score"] > 0]
useful.sort(key=lambda r: r["score"], reverse=True)

print(f"After filtering score=0: {len(useful)}/{len(map_results)} communities are useful.")
print()
for r in useful:
    print(f"  [{r['community_id']}] score={r['score']:>3}")

---
## Step 3 — Reduce: Synthesise a Global Answer  *(Appendix E.4)*

We concatenate the partial answers ranked by helpfulness and ask the LLM to synthesise
them into a single, comprehensive response styled as Markdown.

This is the key insight of GraphRAG: **no single passage in the original text could answer
this question** — the answer emerges from combining community-level knowledge.

The prompt template (Appendix E.4):

In [ ]:
REDUCE_PROMPT = """\
---Goal---
Generate a comprehensive response to the question below by synthesising the
analyst reports provided. Style the response in markdown with clear sections.
Do not simply list the reports — integrate them into a coherent narrative.

---Question---
{question}

---Analyst Reports (ranked by helpfulness, highest first)---
{partial_answers}

Output:"""


# Build the ranked partial-answer context
partial_answers_text = "\n\n".join(
    f"[Report {i+1}, relevance score {r['score']}]\n{r['answer']}"
    for i, r in enumerate(useful)
)

reduce_prompt = REDUCE_PROMPT.format(
    question=QUESTION,
    partial_answers=partial_answers_text,
)

print(f"Reduce context: {len(useful)} partial answers, {len(reduce_prompt):,} chars")
print("Running Reduce step ...")

graphrag_response = llm.invoke([HumanMessage(content=reduce_prompt)])
graphrag_answer = graphrag_response.content

print("Done.")

In [ ]:
display(Markdown("## GraphRAG Answer\n\n" + graphrag_answer))

---
## Comparison — Naive RAG Baseline

To appreciate what GraphRAG adds, we compare it against **naive retrieval-augmented
generation**: retrieve the top-3 most similar text chunks from the original PDF using
TF-IDF, then ask the LLM to answer the question from those chunks alone.

This is the approach standard RAG systems use. It works well for *specific* factual
questions ("Who scored the winning goal?") but struggles with *global* questions like
ours, because no single chunk captures the overall narrative (§1 of the paper).

> **Why TF-IDF and not embeddings?**  
> Embeddings would need a separate model and inference infrastructure. TF-IDF demonstrates
> the same fundamental limitation — surface-level keyword matching — which is the
> contrast the paper argues against. The point is not the retrieval method; it is the
> lack of cross-document synthesis.

In [ ]:
# Re-create chunks from the PDF (same pipeline as Notebook 00, Step 1)
doc = fitz.open(PDF_PATH)
raw_text = "\n".join(page.get_text() for page in doc)
clean_text = re.sub(r"\[[\w,\s]+\]", "", raw_text)
clean_text = re.sub(r"\s{2,}", " ", clean_text)

splitter = RecursiveCharacterTextSplitter(chunk_size=2400, chunk_overlap=400)
chunks = splitter.split_text(clean_text)
print(f"Chunks: {len(chunks)}")

# TF-IDF retrieval
TOP_K = 3
vectorizer = TfidfVectorizer(stop_words="english")
chunk_matrix = vectorizer.fit_transform(chunks)
query_vec    = vectorizer.transform([QUESTION])
scores       = cosine_similarity(query_vec, chunk_matrix)[0]
top_indices  = scores.argsort()[::-1][:TOP_K]

print(f"Top-{TOP_K} chunks by TF-IDF similarity:")
for rank, idx in enumerate(top_indices, 1):
    print(f"  #{rank}  chunk[{idx}]  score={scores[idx]:.3f}  preview: {chunks[idx][:80]}...")

In [ ]:
NAIVE_RAG_PROMPT = """\
Answer the following question using only the provided text passages.
Style the response in markdown.

---Question---
{question}

---Passages---
{passages}

Output:"""

passages_text = "\n\n---\n\n".join(
    f"[Passage {rank+1}]\n{chunks[idx]}" for rank, idx in enumerate(top_indices)
)

naive_response = llm.invoke([
    HumanMessage(content=NAIVE_RAG_PROMPT.format(
        question=QUESTION,
        passages=passages_text,
    ))
])
naive_answer = naive_response.content

display(Markdown("## Naive RAG Answer\n\n" + naive_answer))

---
## Side-by-Side Comparison

In [ ]:
separator = "\n\n---\n\n"

display(Markdown(
    "## GraphRAG Answer\n"
    "*Source: Map-Reduce over 15 community summaries*\n\n"
    + graphrag_answer
    + separator
    + "## Naive RAG Answer\n"
    "*Source: TF-IDF top-3 chunks from the raw PDF*\n\n"
    + naive_answer
))

---
## Summary

| Paper section | What we did |
|---|---|
| § 3.1.6 | Posed a global sensemaking question that no single chunk can answer |
| App. E.4 (Map) | Scored each community summary for helpfulness (0–100) |
| App. E.4 (Map) | Filtered score=0 communities, ranked the rest |
| App. E.4 (Reduce) | Synthesised ranked partial answers into a final markdown response |
| § 1 | Demonstrated the contrast with naive RAG (TF-IDF top-3 chunks) |

### What to look for in the comparison

- **GraphRAG** should cover multiple distinct themes: the Argentina–France final, the
  Qatar infrastructure, the FIFA corruption controversies, the qualification stories.
- **Naive RAG** will anchor on whichever chunks happened to contain the words from the
  question, likely returning a narrower, more factual paragraph.

This is the core argument of the paper (§1): *"queries that require an understanding of
an entire text corpus cannot be effectively addressed by local retrieval alone."*